# Case based on Abadie, Diamond y Hainmueller (2007)
# SYNTHETIC CONTROL METHODS FOR COMPARATIVE CASE STUDIES: ESTIMATING THE EFFECT OF CALIFORNIA'S TOBACCO CONTROL PROGRAM.

https://www.nber.org/system/files/working_papers/t0335/t0335.pdf

### Problem Overview

In 1988, California passed a famous Tobacco Tax and Health Protection Act, which became known as Proposition 99. Its primary effect is to impose a 25-cent per pack state excise tax on the sale of tobacco cigarettes within California, with approximately equivalent excise taxes similarly imposed on the retail sale of other commercial tobacco products, such as cigars and chewing tobacco.

Additional restrictions placed on the sale of tobacco include a ban on cigarette vending machines in public areas accessible by juveniles, and a ban on the individual sale of single cigarettes. Revenue generated by the act was earmarked for various environmental and health care programs, and anti-tobacco advertisements. To evaluate its effect, we can gather data on cigarette sales from multiple states and across a number of years. In our case, we got data from the year 1970 to 2000 from 39 states.

## 1. Import libraries and data:

In [ ]:
# Instalemos Sparse Synthetic Controls
# See: https://github.com/microsoft/SparseSC

import os
install = '"git+https://github.com/microsoft/SparseSC.git"'
os.system(f'pip install -Uqq {install}')

In [ ]:
import pandas as pd
import numpy as np
import SparseSC
from datetime import datetime
import plotly.express as px
import plotly.graph_objects as pgo
pd.set_option("display.max_columns", None)

# Omiting WARNINGS
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Read data

df = pd.read_csv("smoking_data.csv")

df.head()

In [ ]:
#

df.tail()

In [ ]:
#

df.shape

We have data per state as treatment unit and yearly (year column) per-capita sales of cigarettes in packs (cigsale column) and the cigarette retail price (retprice column). We are going to pivot this data so that each row is one treatment unit(state), and columns represent the yearly cigsale value.

In [ ]:
# Pivot Table:

df = df.pivot(index = 'state', columns = 'year', values = "cigsale")

df.head()

In [ ]:
#

df.shape

## 2. Descriptive Analysis

Let's observe how cigarettes sales per capita is trending over time w.r.t California and other states.

In [ ]:
#

plot_df = df.loc[df.index == "California"].T.reset_index(drop = False)

plot_df.head()

In [ ]:
#

plot_df["OtherStates"] = df.loc[df.index != "California"].mean(axis = 0).values

plot_df.head()

In [ ]:
#
fig = px.line( data_frame = plot_df, 
               x = "year", y = ["California","OtherStates"], 
               template = "plotly_white")
# "plotly", "plotly_white", "plotly_dark", "ggplot2", "seaborn", "simple_white", "none"

fig.add_trace( pgo.Scatter( x = [1988, 1988],
                            y = [plot_df.California.min()*0.98, plot_df.OtherStates.max()*1.02],
                            line = { 'dash': 'dash', }, name = 'Proposition 99'
                          ))

fig.update_layout( title = { 'text': "Gap in per-capita cigarette sales(in packs)",
                             'y':0.95, 'x':0.5, },
                   legend = dict(y = 1, x = 0.8, orientation = 'v'),
                   legend_title = "",
                   xaxis_title = "Year", 
                   yaxis_title = "Cigarette Sales Trend",
                   font = dict(size = 15)
                 )

fig.show()

As we can see, there is a general decline in cigarette sales after the 1980s, and with the introduction of Proposition 99, the decreasing trend accelerated for the state of California. 

We cannot say for sure if this is happening with any statistical significance, it is just something we observed by examining the chart above.

To answer the question of whether Proposition 99 influenced cigarette consumption, we will use the pre-intervention period (1970-1988) to build a synthetic control group that mimics California cigarette sales trend. Then, we will see how this synthetic control behaves after the intervention.

## Fitting Synthetic Control using SparseSC package

In [ ]:
# Algunas Funciones previas:

df.iloc[ : , df.columns <= 1988]#.values

In [ ]:
# Algunas funciones previas

df.iloc[ : ,df.columns > 1988]#.values

In [ ]:
# Algunas funciones previas

df.index.values

On a high level SparseSC package provide two functions for fitting Synthetic Controls (SCs) i.e., fit() method and fit_fast() method. 

On a high level:

* fit() - This method tries to compute the weight jointly and results in SCs which are 'optimal'. This is the most common method used in most of the code/libraries I have found but it is computationally expensive and takes a long time to run. Hence, does not scale for larger datasets.

* fit_fast()- This method tries to compute the weight separately by performing some non-matching analysis. This solution is much faster and often the only feasible method with larger datasets. The authors of this package recommend the fit_fast method to start with and only move towards the fit method if needed.

The SparseSC.fit_fast() method required at least three arguments:

- features - This is the NumPy matrix of I/p variables where each row represents a treatment/control unit (states in our case), each column is the period from pre-treatment (1970-1988), and the value in the matrix is the metric of interest (in this case it is the cigsale value)

- targets - This is the NumPy matrix of I/p variables where each row represents a treatment/control unit (states in our case), each column is the period from post-treatment (1999-2000), and the value in the matrix is the metric of interest (in this case it is the cigsale value)

- treatment_units - This is the list of integers containing the row index value of treated units

See: https://github.com/microsoft/SparseSC, for details.

**Note : That treatment units can be a list of multiple treatment indexes. Think of cases where the same treatment is applied to multiple groups, for example, what if proposition 99 was rolled in both California and Minnesota State, in this case, treatment_units will get [2, 15], which are the respective index of these states.**

In [ ]:
## creating required features
features      = df.iloc[ : , df.columns <= 1988].values
targets       = df.iloc[ : , df.columns > 1988].values
treated_units = [idx for idx, val in enumerate(df.index.values) if val == 'California'] # [2]


In [ ]:
#
features.shape

In [ ]:
#
targets.shape

In [ ]:
#
treated_units

In [ ]:
## Fit fast model for fitting Synthetic controls
sc_model = SparseSC.fit_fast( features      = features,
                              targets       = targets,
                              treated_units = treated_units )

Now that we have fitted the model, let's get the Synthetic Control output by using predict() function.

In [ ]:
#

result = df.loc[ df.index == 'California' ].T.reset_index( drop = False )

result.columns = ["year", "Observed"] 

result.head()

In [ ]:
# Weights:

W = sc_model.get_weights( include_trivial_donors = True )

W

In [ ]:
#

W.shape

In [ ]:
# Predict:

result['Synthetic'] = sc_model.predict( df.values )[ treated_units, : ][0]

result.head(5)

Now that we have our synthetic control, we can plot it with the outcome variable of the State of California.

In [ ]:
# Plot:
fig = px.line( data_frame = result, 
               x = "year", y = ["Observed" , "Synthetic"], 
               template = "plotly_white", )

fig.add_trace( pgo.Scatter( x = [1988, 1988],
                            y = [result.Observed.min()*0.98, result.Observed.max()*1.02], 
                            line = { 'dash': 'dash', }, name = 'Proposition 99' 
                          ))

fig.add_trace( pgo.Scatter( x = result['year'],
                            y = plot_df['OtherStates'], name = 'Others (mean)' 
                          ))

fig.update_layout( title = { 'text':"Synthetic Control Assessment", 
                             'y':0.95, 'x':0.5, }, 
                   legend = dict( y = 1, x = 0.8, orientation = 'v'),
                   legend_title = "",
                   xaxis_title = "Year", yaxis_title = "Per-capita cigarette sales (in packs)",
                   font = dict( size = 15 ) )

fig.show()

In the pre-intervention period, the synthetic control does not reproduce the treatment exactly but follows the curve closely. This is a good sign, as it indicates that we are not overfitting. Also, note that we do see divergence after the intervention (introduction of Proposition 99) after 1988.

With the synthetic control groups in hand, we can estimate the treatment effect as the gap between the treated and the synthetic control outcomes.

In [ ]:
# Fig - Gap in Per-capita cigarette sales in California w.r.t Synthetic Control

result['California Effect'] = result.Observed - result.Synthetic

fig = px.line( data_frame = result, 
               x = "year", y = "California Effect", 
               template = "plotly_white",)

fig.add_hline(0)

fig.add_trace( pgo.Scatter( 
               x = [ 1988, 1988 ], 
               y = [result["California Effect"].min()*0.98, result["California Effect"].max()*1.02], 
               line = { 'dash': 'dash', }, name='Proposition 99' ))

fig.update_layout( title = { 'text':"Difference across time", 
                             'y':0.95, 'x':0.5, },
                             legend =  dict(y = 1, x = 0.8, orientation = 'v'),
                             legend_title = "",
                             xaxis_title = "Year", 
                             yaxis_title = "Gap in Per-capita cigarette sales (in packs)",
                             font = dict( size = 15 ))

fig.show()

In [ ]:
print( f"Effect of Proposition 99 w.r.t Synthetic Control => \
       {np.round(result.loc[result.year == 2000, 'California Effect'].values[0],1)} packs")
